In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
# saving as a file
df.write.format("delta").mode("overwrite").save("/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta")

In [0]:
# saving as table without location
df.write.format("delta").mode("overwrite").saveAsTable("mydb.employees_delta_managed")


In [0]:
emp_delta_loc = "/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta/"

In [0]:
# reading delta table using 
emp_delta_df = spark.read.format("delta").option("inferSchema", "true").load(emp_delta_loc)

emp_delta_df.display()

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
101,Ayush,Verma,IT,100000
102,Neena,Kochhar,HR,17000
103,Lex,De Haan,Finance,17000
104,Alexander,Hunold,IT,9000
105,Bruce,Ernst,IT,6000
106,David,Austin,Sales,4800
107,Valli,Pataballa,Finance,4800
108,Diana,Lorentz,Marketing,4200


In [0]:
# reading delta table using 
emp_delta_df = spark.read.format("delta").option("inferSchema", "true").option("versionAsOf", 5).load(emp_delta_loc)

emp_delta_df.display()

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
101,Ayush,Verma,IT,100000
102,Neena,Kochhar,HR,17000
103,Lex,De Haan,Finance,17000
104,Alexander,Hunold,IT,9000
105,Bruce,Ernst,IT,6000
106,David,Austin,Sales,4800
107,Valli,Pataballa,Finance,4800
108,Diana,Lorentz,Marketing,4200
109,Nancy,Greenberg,Finance,12000


In [0]:
# updating delta table using pyspark 
emp_delta_table = DeltaTable.forPath(spark, emp_delta_loc)

emp_delta_table.update(
    condition=col("EMPLOYEE_ID") == 101,
    set={
        "FIRST_NAME": lit("Ayush"),
        "SALARY": lit(100000)
    }
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6789803789550944>, line 4
      1 # updating delta table using pyspark 
      2 emp_delta_table = DeltaTable.forPath(spark, emp_delta_loc)
----> 4 emp_delta_table.update(
      5     condition=col("EMPLOYEE_ID") == 101,
      6     set={"LAST_NAME": "Verma"}
      7 )

File /databricks/python/lib/python3.12/site-packages/delta/connect/tables.py:153, in DeltaTable.update(self, condition, set)
    147 plan = UpdateTable(
    148     self._plan,
    149     condition,
    150     assignments
    151 )
    152 df = DataFrame(plan, session=self._spark)
--> 153 return self._spark.createDataFrame(df.toPandas())

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1943, in DataFrame.toPandas(self)
   1941 def toPandas(self) -> "PandasDataFrameLike":
   1942     query = self._plan.to_proto(self._se

In [0]:
emp_delta_table.delete(condition=col("EMPLOYEE_ID") == 110)

DataFrame[num_affected_rows: bigint]

In [0]:
%sql 
-- reading using sql 
select * from delta.`/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta/`


EMPLOYEE_ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
101,Ayush,King,IT,100000
102,Neena,Kochhar,HR,17000
103,Lex,De Haan,Finance,17000
104,Alexander,Hunold,IT,9000
105,Bruce,Ernst,IT,6000
106,David,Austin,Sales,4800
107,Valli,Pataballa,Finance,4800
108,Diana,Lorentz,Marketing,4200
109,Nancy,Greenberg,Finance,12000


In [0]:
%sql
-- update using sql
update delta.`/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta/`
set LAST_NAME = 'Verma'
where EMPLOYEE_ID = 101

num_affected_rows
1


In [0]:
%sql
-- delete delta table using sql
delete from delta.`/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta/`
where EMPLOYEE_ID = 109

num_affected_rows
1


In [0]:
%sql
-- checking history
DESCRIBE HISTORY  delta.`/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-07-14T07:21:25.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),9343985d-b335-4bc5-ba18-f961b59800d2,0714-064031-2fufg8in-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2036, p25FileSize -> 2027, numDeletionVectorsRemoved -> 1, minFileSize -> 2027, numAddedFiles -> 1, maxFileSize -> 2027, p75FileSize -> 2027, p50FileSize -> 2027, numAddedBytes -> 2027)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-07-14T07:21:23.000Z,7096814250910626,ayushhathiwan2@gmail.com,DELETE,"Map(predicate -> [""(EMPLOYEE_ID#13455L = 109)""])",null,List(2667370519330194),9343985d-b335-4bc5-ba18-f961b59800d2,0714-064031-2fufg8in-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1677, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1213, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 464)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-07-14T07:20:18.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),339d26d8-36fb-4ee1-af09-c82d9e76d049,0714-064031-2fufg8in-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3730, p25FileSize -> 2036, numDeletionVectorsRemoved -> 1, minFileSize -> 2036, numAddedFiles -> 1, maxFileSize -> 2036, p75FileSize -> 2036, p50FileSize -> 2036, numAddedBytes -> 2036)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-07-14T07:20:16.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(EMPLOYEE_ID#13048L = 101)""])",null,List(2667370519330194),339d26d8-36fb-4ee1-af09-c82d9e76d049,0714-064031-2fufg8in-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2708, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1160, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1686, rewriteTimeMs -> 1535)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-07-14T07:10:50.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),2699fb51-21e5-4d42-a95a-d17efcbd198b,0714-064031-2fufg8in-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3743, p25FileSize -> 2044, numDeletionVectorsRemoved -> 1, minFileSize -> 2044, numAddedFiles -> 1, maxFileSize -> 2044, p75FileSize -> 2044, p50FileSize -> 2044, numAddedBytes -> 2044)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-14T07:10:47.000Z,7096814250910626,ayushhathiwan2@gmail.com,DELETE,"Map(predicate -> [""(EMPLOYEE_ID#12520L = 110)""])",null,List(2667370519330194),2699fb51-21e5-4d42-a95a-d17efcbd198b,0714-064031-2fufg8in-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1612, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 1130, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 481)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-14T07:08:36.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(EMPLOYEE_ID#12208L = 101)""])",null,List(2667370519330194),6050c4fc-cbc9-46b4-91f0-aeccc98f2462,0714-064031-2fufg8in-v2n,0,WriteSerializable,false,"Map(numRemo

In [0]:
emp_delta_table.history().select("*").display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-07-14T07:21:25.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),9343985d-b335-4bc5-ba18-f961b59800d2,0714-064031-2fufg8in-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2036, p25FileSize -> 2027, numDeletionVectorsRemoved -> 1, minFileSize -> 2027, numAddedFiles -> 1, maxFileSize -> 2027, p75FileSize -> 2027, p50FileSize -> 2027, numAddedBytes -> 2027)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-07-14T07:21:23.000Z,7096814250910626,ayushhathiwan2@gmail.com,DELETE,"Map(predicate -> [""(EMPLOYEE_ID#13455L = 109)""])",null,List(2667370519330194),9343985d-b335-4bc5-ba18-f961b59800d2,0714-064031-2fufg8in-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1677, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1213, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 464)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-07-14T07:20:18.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),339d26d8-36fb-4ee1-af09-c82d9e76d049,0714-064031-2fufg8in-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3730, p25FileSize -> 2036, numDeletionVectorsRemoved -> 1, minFileSize -> 2036, numAddedFiles -> 1, maxFileSize -> 2036, p75FileSize -> 2036, p50FileSize -> 2036, numAddedBytes -> 2036)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-07-14T07:20:16.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(EMPLOYEE_ID#13048L = 101)""])",null,List(2667370519330194),339d26d8-36fb-4ee1-af09-c82d9e76d049,0714-064031-2fufg8in-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2708, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1160, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1686, rewriteTimeMs -> 1535)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-07-14T07:10:50.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),2699fb51-21e5-4d42-a95a-d17efcbd198b,0714-064031-2fufg8in-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3743, p25FileSize -> 2044, numDeletionVectorsRemoved -> 1, minFileSize -> 2044, numAddedFiles -> 1, maxFileSize -> 2044, p75FileSize -> 2044, p50FileSize -> 2044, numAddedBytes -> 2044)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-14T07:10:47.000Z,7096814250910626,ayushhathiwan2@gmail.com,DELETE,"Map(predicate -> [""(EMPLOYEE_ID#12520L = 110)""])",null,List(2667370519330194),2699fb51-21e5-4d42-a95a-d17efcbd198b,0714-064031-2fufg8in-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1612, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 1130, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 481)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-14T07:08:36.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(EMPLOYEE_ID#12208L = 101)""])",null,List(2667370519330194),6050c4fc-cbc9-46b4-91f0-aeccc98f2462,0714-064031-2fufg8in-v2n,0,WriteSerializable,false,"Map(numRemo

In [0]:
emp_delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-07-14T07:21:25.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),9343985d-b335-4bc5-ba18-f961b59800d2,0714-064031-2fufg8in-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2036, p25FileSize -> 2027, numDeletionVectorsRemoved -> 1, minFileSize -> 2027, numAddedFiles -> 1, maxFileSize -> 2027, p75FileSize -> 2027, p50FileSize -> 2027, numAddedBytes -> 2027)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-07-14T07:21:23.000Z,7096814250910626,ayushhathiwan2@gmail.com,DELETE,"Map(predicate -> [""(EMPLOYEE_ID#13455L = 109)""])",null,List(2667370519330194),9343985d-b335-4bc5-ba18-f961b59800d2,0714-064031-2fufg8in-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1677, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1213, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 464)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-07-14T07:20:18.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),339d26d8-36fb-4ee1-af09-c82d9e76d049,0714-064031-2fufg8in-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3730, p25FileSize -> 2036, numDeletionVectorsRemoved -> 1, minFileSize -> 2036, numAddedFiles -> 1, maxFileSize -> 2036, p75FileSize -> 2036, p50FileSize -> 2036, numAddedBytes -> 2036)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-07-14T07:20:16.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(EMPLOYEE_ID#13048L = 101)""])",null,List(2667370519330194),339d26d8-36fb-4ee1-af09-c82d9e76d049,0714-064031-2fufg8in-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2708, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1160, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1686, rewriteTimeMs -> 1535)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-07-14T07:10:50.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),2699fb51-21e5-4d42-a95a-d17efcbd198b,0714-064031-2fufg8in-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3743, p25FileSize -> 2044, numDeletionVectorsRemoved -> 1, minFileSize -> 2044, numAddedFiles -> 1, maxFileSize -> 2044, p75FileSize -> 2044, p50FileSize -> 2044, numAddedBytes -> 2044)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-14T07:10:47.000Z,7096814250910626,ayushhathiwan2@gmail.com,DELETE,"Map(predicate -> [""(EMPLOYEE_ID#12520L = 110)""])",null,List(2667370519330194),2699fb51-21e5-4d42-a95a-d17efcbd198b,0714-064031-2fufg8in-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1612, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 1130, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 481)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-14T07:08:36.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(EMPLOYEE_ID#12208L = 101)""])",null,List(2667370519330194),6050c4fc-cbc9-46b4-91f0-aeccc98f2462,0714-064031-2fufg8in-v2n,0,WriteSerializable,false,"Map(numRemo

In [0]:
%sql 
SELECT * FROM delta.`/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta/`


EMPLOYEE_ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
101,Ayush,Verma,IT,100000
102,Neena,Kochhar,HR,17000
103,Lex,De Haan,Finance,17000
104,Alexander,Hunold,IT,9000
105,Bruce,Ernst,IT,6000
106,David,Austin,Sales,4800
107,Valli,Pataballa,Finance,4800
108,Diana,Lorentz,Marketing,4200


In [0]:
%sql
RESTORE TABLE delat.`/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta/` TO VERSION AS OF 1

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8337947717929174>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'RESTORE TABLE delat.`/Volumes/workspace/mydb/myvolume/delta_lake/employees_delta/` TO VERSION AS OF 1\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 exc

In [0]:
%sql
select * from mydb.employees_delta_managed;

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
101,Steven,King,IT,25000
102,Neena,Kochhar,HR,17000
103,Lex,De Haan,Finance,17000
104,Alexander,Hunold,IT,9000
105,Bruce,Ernst,IT,6000
106,David,Austin,Sales,4800
107,Valli,Pataballa,Finance,4800
108,Diana,Lorentz,Marketing,4200
109,Nancy,Greenberg,Finance,12000
110,John,Chen,Sales,8200


In [0]:
%sql 
update mydb.employees_delta_managed
SET first_name = 'Ayush'
where employee_id = 101

num_affected_rows
1


In [0]:
%sql 
desc history mydb.employees_delta_managed

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-07-14T08:08:14.000Z,7096814250910626,ayushhathiwan2@gmail.com,RESTORE,"Map(version -> 1, timestamp -> null)",null,List(2667370519330194),ae52170c-358e-41f4-878d-9606a5915609,0714-080340-2qmqu4y3-v2n,2,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 3749, numRemovedFiles -> 2, restoredFilesSize -> 2029, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 1, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 2029)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-14T08:06:17.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(employee_id#11594L = 101)""])",null,List(2667370519330194),bd7f835c-1f29-4819-af6c-3ac7a123bab2,0714-080340-2qmqu4y3-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 6270, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2324, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1720, rewriteTimeMs -> 3911)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-14T08:06:02.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(employee_id#11480L = 1)""])",null,List(2667370519330194),cdf3b346-3218-4346-8ad0-6601997418be,0714-080340-2qmqu4y3-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1112, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1065, numAddedFiles -> 0, numUpdatedRows -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-14T06:56:48.000Z,7096814250910626,ayushhathiwan2@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2667370519330194),34efdda0-13ca-4318-9f79-b6f37539c887,0714-064031-2fufg8in-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 2029)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%sql
-- restore to version
RESTORE TABLE mydb.employees_delta_managed TO VERSION AS OF 1;

-- restore to timestamp
RESTORE TABLE mydb.employees_delta_managed TO TIMESTAMP AS OF '2026-07-14T08:08:14.000+00:00';

-- now table will restore to version 1 and we will get the previous data
select * from mydb.employees_delta_managed;

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
101,Steven,King,IT,25000
102,Neena,Kochhar,HR,17000
103,Lex,De Haan,Finance,17000
104,Alexander,Hunold,IT,9000
105,Bruce,Ernst,IT,6000
106,David,Austin,Sales,4800
107,Valli,Pataballa,Finance,4800
108,Diana,Lorentz,Marketing,4200
109,Nancy,Greenberg,Finance,12000
110,John,Chen,Sales,8200


In [0]:
%sql
select * from mydb.employees_delta_managed;

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
101,Steven,King,IT,25000
102,Neena,Kochhar,HR,17000
103,Lex,De Haan,Finance,17000
104,Alexander,Hunold,IT,9000
105,Bruce,Ernst,IT,6000
106,David,Austin,Sales,4800
107,Valli,Pataballa,Finance,4800
108,Diana,Lorentz,Marketing,4200
109,Nancy,Greenberg,Finance,12000
110,John,Chen,Sales,8200


DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [0]:
emp_delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-07-14T08:15:20.000Z,7096814250910626,ayushhathiwan2@gmail.com,RESTORE,"Map(version -> 3, timestamp -> null)",null,List(2667370519330194),7b061ae7-cd71-4bc5-b465-25a26b7208d9,0714-080340-2qmqu4y3-v2n,7,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 2027, numRemovedFiles -> 1, restoredFilesSize -> 2044, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 2044)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
7,2026-07-14T07:21:25.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),9343985d-b335-4bc5-ba18-f961b59800d2,0714-064031-2fufg8in-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2036, p25FileSize -> 2027, numDeletionVectorsRemoved -> 1, minFileSize -> 2027, numAddedFiles -> 1, maxFileSize -> 2027, p75FileSize -> 2027, p50FileSize -> 2027, numAddedBytes -> 2027)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-07-14T07:21:23.000Z,7096814250910626,ayushhathiwan2@gmail.com,DELETE,"Map(predicate -> [""(EMPLOYEE_ID#13455L = 109)""])",null,List(2667370519330194),9343985d-b335-4bc5-ba18-f961b59800d2,0714-064031-2fufg8in-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1677, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1213, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 464)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-07-14T07:20:18.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),339d26d8-36fb-4ee1-af09-c82d9e76d049,0714-064031-2fufg8in-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3730, p25FileSize -> 2036, numDeletionVectorsRemoved -> 1, minFileSize -> 2036, numAddedFiles -> 1, maxFileSize -> 2036, p75FileSize -> 2036, p50FileSize -> 2036, numAddedBytes -> 2036)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-07-14T07:20:16.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(EMPLOYEE_ID#13048L = 101)""])",null,List(2667370519330194),339d26d8-36fb-4ee1-af09-c82d9e76d049,0714-064031-2fufg8in-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2708, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1160, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1686, rewriteTimeMs -> 1535)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-07-14T07:10:50.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),2699fb51-21e5-4d42-a95a-d17efcbd198b,0714-064031-2fufg8in-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3743, p25FileSize -> 2044, numDeletionVectorsRemoved -> 1, minFileSize -> 2044, numAddedFiles -> 1, maxFileSize -> 2044, p75FileSize -> 2044, p50FileSize -> 2044, numAddedBytes -> 2044)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-14T07:10:47.000Z,7096814250910626,ayushhathiwan2@gmail.com,DELETE,"Map(predicate -> [""(EMPLOYEE_ID#12520L = 110)""])",null,List(2667370519330194),2699fb51-21e5-4d42-a95a-d17efcbd198b,0714-064031-2fufg8in-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemov

In [0]:
data = [(1, "A"), (2, "B"), (3, "C")]

df1 = spark.createDataFrame(data, schema=["id", "name"])

df1.write.format("delta").mode("append")\
    .save("/Volumes/workspace/mydb/myvolume/delta_lake/schema_evolution_delta")


In [0]:
data = [4, 5, 6]

# schema is changed in df2
df2 = spark.createDataFrame(data, schema=["age"])

# .enabling overwriteSchema option to true for schema evolution
df2.write.format("delta").mode("overwrite")\
    .option("overwriteSchema", True)\
    .save("/Volumes/workspace/mydb/myvolume/delta_lake/schema_evolution_delta")

In [0]:
%sql
select * from delta.`/Volumes/workspace/mydb/myvolume/delta_lake/schema_evolution_delta`

age
4
5
6


In [0]:
%sql
CREATE TABLE mydb.copy_into_person_table (
    id INT, 
    name STRING
)

USING Delta;

In [0]:
%sql 
COPY INTO mydb.copy_into_person_table
FROM '/Volumes/workspace/mydb/myvolume/delta_lake/copy_into_person/person.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

id,name
1,Ayush
2,Aman
3,Aryan


In [0]:
data = [(1, 'Ayush', 'India'), (2, 'John', 'USA'), (3, 'Ravi', 'India')]

df = spark.createDataFrame(data, schema=["id", "name", "country"])

df.write.csv('/Volumes/workspace/mydb/myvolume/person_src/person.csv', mode='overwrite', header=True)

In [0]:
%sql
CREATE TABLE mydb.merge_into_person_table (
    id INT,
    name STRING,
    country STRING
)
USING Delta;

In [0]:
df = spark.read.csv('/Volumes/workspace/mydb/myvolume/person_src/person.csv', header=True)

df.createTempView("person_source")

In [0]:
%sql 
MERGE INTO mydb.merge_into_person_table as target
USING person_source as source
ON target.id = source.id
WHEN MATCHED THEN
    UPDATE SET target.name = source.name, target.country = source.country
WHEN NOT MATCHED THEN
    INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


In [0]:
%sql
select * from mydb.merge_into_person_table;

id,name,country
1,Ayush,India
3,Ravi,India
2,John,USA


In [0]:
person_dlt = DeltaTable.forName(spark, "mydb.merge_into_person_table")

In [0]:
person_dlt.alias('t').merge(
    source=df.alias('s'), 
    condition="s.id = t.id"
).whenMatchedUpdate(set={'name': "s.name", 'country': "s.country"})\
.whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
help(person_dlt.merge)

Help on method merge in module delta.connect.tables:

merge(source: pyspark.sql.connect.dataframe.DataFrame, condition: Union[str, pyspark.sql.connect.column.Column]) -> 'DeltaMergeBuilder' method of delta.connect.tables.DeltaTable instance
    Merge data from the `source` DataFrame based on the given merge `condition`. This returns
    a :class:`DeltaMergeBuilder` object that can be used to specify the update, delete, or
    insert actions to be performed on rows based on whether the rows matched the condition or
    not. See :class:`DeltaMergeBuilder` for a full description of this operation and what
    combinations of update, delete and insert operations are allowed.

    Example 1 with conditions and update expressions as SQL formatted string::

        deltaTable.alias("events").merge(
            source = updatesDF.alias("updates"),
            condition = "events.eventId = updates.eventId"
          ).whenMatchedUpdate(set =
            {
              "data": "updates.data",
 

In [0]:
students = [
    (1, "Aman", 85),
    (2, "Riya", 92),
    (3, "Rahul", 78),
    (4, "Sneha", 88),
    (5, "Karan", 67),
    (6, "Priya", 95),
    (7, "Arjun", 73),
    (8, "Neha", 81),
    (9, "Vikram", 90),
    (10, "Anjali", 76)
]

# Column names
columns = ["id", "name", "marks"]

# Create DataFrame
df = spark.createDataFrame(students, columns)

df = df.repartition(3)

df.write.mode("overwrite").save("/Volumes/workspace/mydb/myvolume/delta_lake/students_delta")

In [0]:
%sql 
select * from delta.`/Volumes/workspace/mydb/myvolume/delta_lake/students_delta`

id,name,marks
3,Rahul,78
6,Priya,95
7,Arjun,73
9,Vikram,90
1,Aman,85
4,Sneha,88
8,Neha,81
10,Anjali,76
2,Riya,92
5,Karan,67


In [0]:
%sql
update delta.`/Volumes/workspace/mydb/myvolume/delta_lake/students_delta`
set marks = 60
where id = 10

num_affected_rows
1


In [0]:
%sql
describe history delta.`/Volumes/workspace/mydb/myvolume/delta_lake/students_delta`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-18T04:16:35.000Z,7096814250910626,ayushhathiwan2@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2667370519330194),b87361ae-134b-489e-a50b-f94fb89a0ae9,0718-025451-1geoabwo-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 5124, p25FileSize -> 1409, numDeletionVectorsRemoved -> 1, minFileSize -> 1409, numAddedFiles -> 1, maxFileSize -> 1409, p75FileSize -> 1409, p50FileSize -> 1409, numAddedBytes -> 1409)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-18T04:16:33.000Z,7096814250910626,ayushhathiwan2@gmail.com,UPDATE,"Map(predicate -> [""(id#13011L = 1)""])",null,List(2667370519330194),b87361ae-134b-489e-a50b-f94fb89a0ae9,0718-025451-1geoabwo-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2442, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1115, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1237, rewriteTimeMs -> 1324)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-18T03:49:20.000Z,7096814250910626,ayushhathiwan2@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2667370519330194),ccdc4c7d-4cd8-4507-bbbe-83b44cdf7834,0718-025451-1geoabwo-v2n,null,WriteSerializable,false,"Map(numFiles -> 3, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 3887)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
df = spark.createDataFrame(dbutils.fs.ls("/Volumes/workspace/mydb/myvolume/delta_lake/students_delta"), schema=["file", "a", "c", "d"])

df.display()

file,a,c,d
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/_delta_log/,_delta_log/,0,1784349216610
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/deletion_vector_1b62ca70-8a80-4b35-864d-15bcbc0d589e.bin,deletion_vector_1b62ca70-8a80-4b35-864d-15bcbc0d589e.bin,43,1784348191000
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/deletion_vector_2a7912be-6e08-4a9d-8737-c63442e8528f.bin,deletion_vector_2a7912be-6e08-4a9d-8737-c63442e8528f.bin,87,1784349205000
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/deletion_vector_63e92523-e5aa-49fd-9c9c-d1eabe5f8047.bin,deletion_vector_63e92523-e5aa-49fd-9c9c-d1eabe5f8047.bin,43,1784348983000
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/part-00000-09999a53-7691-4576-97a8-546551587c26.c000.snappy.parquet,part-00000-09999a53-7691-4576-97a8-546551587c26.c000.snappy.parquet,1307,1784346559000
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/part-00000-7975c665-d48b-452b-9b11-25a1114f6838.c000.snappy.parquet,part-00000-7975c665-d48b-452b-9b11-25a1114f6838.c000.snappy.parquet,1407,1784349208000
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/part-00000-9a623d81-bb8e-48cf-9a27-1bb96533ac41.c000.snappy.parquet,part-00000-9a623d81-bb8e-48cf-9a27-1bb96533ac41.c000.snappy.parquet,1242,1784348984000
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/part-00000-9dd450d3-1cb9-4200-8dd0-31553b190e58.c000.snappy.parquet,part-00000-9dd450d3-1cb9-4200-8dd0-31553b190e58.c000.snappy.parquet,1237,1784348192000
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/part-00000-ce56cb96-26c9-4f44-9a24-ceee98a4bbfe.c000.snappy.parquet,part-00000-ce56cb96-26c9-4f44-9a24-ceee98a4bbfe.c000.snappy.parquet,1246,1784349206000
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/part-00000-f1b8bcbb-46ed-4b7f-a127-d93378235041.c000.snappy.parquet,part-00000-f1b8bcbb-46ed-4b7f-a127-d93378235041.c000.snappy.parquet,1409,1784348195000


In [0]:
dbutils.fs.ls("/Volumes/workspace/mydb/myvolume/delta_lake/students_delta")

[FileInfo(path='dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/_delta_log/', name='_delta_log/', size=0, modificationTime=1784349077734),
 FileInfo(path='dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/deletion_vector_1b62ca70-8a80-4b35-864d-15bcbc0d589e.bin', name='deletion_vector_1b62ca70-8a80-4b35-864d-15bcbc0d589e.bin', size=43, modificationTime=1784348191000),
 FileInfo(path='dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/deletion_vector_63e92523-e5aa-49fd-9c9c-d1eabe5f8047.bin', name='deletion_vector_63e92523-e5aa-49fd-9c9c-d1eabe5f8047.bin', size=43, modificationTime=1784348983000),
 FileInfo(path='dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/part-00000-09999a53-7691-4576-97a8-546551587c26.c000.snappy.parquet', name='part-00000-09999a53-7691-4576-97a8-546551587c26.c000.snappy.parquet', size=1307, modificationTime=1784346559000),
 FileInfo(path='dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta/par

In [0]:
%sql
optimize delta.`/Volumes/workspace/mydb/myvolume/delta_lake/students_delta`

path,metrics
dbfs:/Volumes/workspace/mydb/myvolume/delta_lake/students_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1784355026051, 1784355030551, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
deltaTable = DeltaTable.forPath(spark, "/path/to/table")

deltaTable.optimize().executeZOrderBy("id", "department")

In [0]:
%sql
ZORDER BY (department, salary);